In [1]:
from huggingface_hub import login

login()

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


In [2]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorForLanguageModeling

model_name = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
dataset = load_dataset("karthiksagarn/astro_horoscope", split="train")

def tokenize(batch):
    return tokenizer(
        batch["horoscope"],
        truncation=True,
        max_length=512,
    )

dataset = dataset.map(tokenize, batched=True, remove_columns=dataset.column_names)
dataset = dataset.train_test_split(test_size=0.1)

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

README.md:   0%|          | 0.00/31.0 [00:00<?, ?B/s]

horoscope.csv:   0%|          | 0.00/9.59M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/21959 [00:00<?, ? examples/s]

Map:   0%|          | 0/21959 [00:00<?, ? examples/s]

In [3]:
data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

In [4]:
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer

model_name = "Qwen/Qwen3-0.6B"
model = AutoModelForCausalLM.from_pretrained(model_name, dtype="auto")

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.50GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [ ]:
# training_args = TrainingArguments(
#     output_dir="qwen3-finetuned",
#     num_train_epochs=3,
#     per_device_train_batch_size=2,
#     gradient_accumulation_steps=8,
#     gradient_checkpointing=True,
#     gradient_checkpointing_kwargs={"every_n_layers": 4},
#     bf16=True,
#     learning_rate=2e-5,
#     logging_steps=10,
#     eval_strategy="epoch",
#     save_strategy="epoch",
#     load_best_model_at_end=True,
# )

In [ ]:
# significantly faster training with larger batch size, but requires more GPU memory
# 1.5m was half done
training_args = TrainingArguments(
    output_dir="qwen3-finetuned",
    num_train_epochs=1,                 # raise once you've confirmed it works
    per_device_train_batch_size=16,     # drop to 8 if you hit OOM
    gradient_accumulation_steps=1,
    bf16=True,                          # only on L4/A100
    dataloader_num_workers=2,
    learning_rate=2e-5,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)


In [9]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
)

trainer.train()


Epoch,Training Loss,Validation Loss
1,2.394252,2.516465


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


TrainOutput(global_step=1236, training_loss=2.553204598164481, metrics={'train_runtime': 247.194, 'train_samples_per_second': 79.949, 'train_steps_per_second': 5.0, 'total_flos': 6659899648376832.0, 'train_loss': 2.553204598164481, 'epoch': 1.0})

In [10]:
trainer.push_to_hub()

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

CommitInfo(commit_url='https://huggingface.co/aricode00x/qwen3-finetuned/commit/fa656d645b6a9c5139ee8382e33e9e810daf213a', commit_message='End of training', commit_description='', oid='fa656d645b6a9c5139ee8382e33e9e810daf213a', pr_url=None, repo_url=RepoUrl('https://huggingface.co/aricode00x/qwen3-finetuned', endpoint='https://huggingface.co', repo_type='model', repo_id='aricode00x/qwen3-finetuned'), pr_revision=None, pr_num=None)

# Inference Test via Free Pipeline

In [11]:
trainer.model

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layer

In [12]:
from transformers import pipeline

# Simplest test, right after training (the model and tokenizer are already in memory):
pipe = pipeline("text-generation", model=trainer.model, tokenizer=tokenizer)

# aricode00x/qwen3-finetuned
# If you restart the runtime later, load from the Hub instead. Replace the repo id with the one you pushed to:
# pipe = pipeline("text-generation", model="your-username/qwen3-finetuned", device_map="auto")


out = pipe(
    "Aries:",            # use the same style of prompt your training data starts with
    max_new_tokens=100,
    do_sample=True,
    temperature=0.8,
    top_p=0.9,
)
print(out[0]["generated_text"])


[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'top_p', 'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Aries: The day's planetary energy brings with it a feeling of energy and excitement. You are likely to feel as if you have an extra spark in your system, and this will make you excited about anything that involves your body. Don't be surprised if you begin to feel a little out of control, and you might want to take some time out for yourself to recover. Don't be afraid to take a break from the situation that is making you feel a little out of balance. Relax, be relaxed,


## Compare

In [13]:
from transformers import pipeline

prompts = [
    "Aries:",
    "Today for Taurus,",
    "Your horoscope for Gemini:",
]

base = pipeline("text-generation", model="Qwen/Qwen3-0.6B", device_map="auto")
tuned = pipeline("text-generation", model=trainer.model, tokenizer=tokenizer)

for p in prompts:
    for name, pipe in [("BASE", base), ("TUNED", tuned)]:
        out = pipe(p, max_new_tokens=80, do_sample=False, return_full_text=False)
        print(f"[{name}] {p}\n{out[0]['generated_text']}\n")
    print("-" * 60)


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[BASE] Aries:
 Aries is a constellation in the northern sky, located in the northern hemisphere. It is the first of the 12 zodiac signs, and is the first of the 12 zodiac signs. Aries is the first of the 12 zodiac signs, and is the first of the 12 zodiac signs. Aries is the first of the 12 z



[transformers] Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[TUNED] Aries:
 You're going to be very busy today. You may have to work overtime to get things done. You may have to do a lot of work that you don't normally do. You may have to do a lot of work that you don't normally do. You may have to do a lot of work that you don't normally do. You may have to do a lot of work that you don

------------------------------------------------------------


...

[BASE] Today for Taurus,
 I'm going to be writing about the importance of self-compassion. I know that self-compassion is a very important topic, but I want to make sure that I'm not just going off the rails. I want to make sure that I'm writing about it in a way that's both insightful and practical. I want to make sure that I'm not just going off the rails.



[transformers] Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[TUNED] Today for Taurus,
 you may be feeling a bit like a "gambler" who is trying to win a big prize. You may be trying to win a big prize by winning a big amount of money. You may be trying to win a big amount of money by winning a big amount of money. You may be trying to win a big amount of money by winning a big amount of money. You may be

------------------------------------------------------------


[transformers] Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[BASE] Your horoscope for Gemini:
 A New Era of Prosperity and Growth

As the sun sets and the stars align, the horoscope for Gemini is a beacon of hope and guidance. In this new era, Gemini is poised to thrive, with opportunities that are both promising and abundant. The sun's rays are warm and inviting, symbolizing the energy that Gemini brings to the table. With their natural affinity for communication and creativity,

[TUNED] Your horoscope for Gemini:
 You may be feeling a bit overwhelmed by the current planetary energy. You may feel like you have to do a lot of work, and you may feel like you have to do it all. You may feel like you have to do it all. You may feel like you have to do it all. You may feel like you have to do it all. You may feel like you have to do it

------------------------------------------------------------
